# B08C — Efisiensi inferensi Teacher dan Student KD

Profiling **VGG16, DenseNet121, MobileNetV2, dan NASNetMobile**, mengikuti arsitektur final B04–B06. Tidak melatih model, tidak mengevaluasi ulang test, dan tidak mengubah hasil penelitian lama.

**Jalankan dalam runtime Colab GPU baru**, idealnya L4. Jalankan Cell 1–7 berurutan. Definisi arsitektur sudah disertakan; checkpoint perwakilan CBIS-DDSM seed 42 dimuat dari Google Drive. Tidak perlu mengekstrak dataset atau menjalankan B05/B06 sebelumnya.

Hasil: parameter, MACs Conv/Dense, ukuran bobot, latensi batch 1, throughput batch 8, memori aktif TensorFlow; CSV, Excel, dan ZIP laporan. Angka baru muncul setelah notebook dijalankan.

Kontrol/Hinton/adaptasi SSID-KD memakai arsitektur deployment yang sama untuk masing-masing Student. Projector dan Teacher tidak ikut inferensi Student. Profiling satu arsitektur tidak membuktikan KD meningkatkan kecepatan dibanding kontrol.

## Cell 1 — Siapkan dependensi dan Google Drive
Parent notebook sengaja tidak mengimpor TensorFlow. Pengukuran berjalan pada subprocess terpisah agar model lain dan optimizer tidak mencemari peak memory. Instalasi hanya mencocokkan `tf-keras` dengan versi TensorFlow Colab, bukan mengganti TensorFlow.

In [ ]:
import sys, os, json, hashlib, shutil, subprocess, zipfile, importlib.metadata as metadata
from pathlib import Path
from datetime import datetime, timezone
import uuid

if 'tensorflow' in sys.modules or 'tf_keras' in sys.modules:
    raise RuntimeError('Gunakan runtime baru/restart session, lalu mulai Cell 1. Parent harus bebas alokasi TensorFlow.')
try:
    TF_VERSION = metadata.version('tensorflow')
except metadata.PackageNotFoundError:
    raise RuntimeError('TensorFlow tidak ditemukan. Jalankan pada Google Colab GPU standar.')
TF_MINOR = '.'.join(TF_VERSION.split('.')[:2])
try:
    KERAS_VERSION = metadata.version('tf-keras')
except metadata.PackageNotFoundError:
    KERAS_VERSION = ''
if '.'.join(KERAS_VERSION.split('.')[:2]) != TF_MINOR:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',f'tf-keras~={TF_MINOR}.0'])
try:
    import openpyxl
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','openpyxl'])
import numpy as np
import pandas as pd
from google.colab import drive, files
from IPython.display import display
drive.mount('/content/drive')
print('TensorFlow:',TF_VERSION,'| tf-keras:',metadata.version('tf-keras'))
print(subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'],text=True))

## Cell 2 — Konfigurasi pengukuran
Default: input 512 × 512, float32 dengan TF32 dimatikan, batch 1 dan 8, 30 warm-up, 5 putaran × 50 panggilan. Gunakan konfigurasi sama untuk semua model. Jika batch 8 OOM, ubah menjadi `[1, 4]` dan jalankan ulang **semua model** dalam satu run baru.

Output memakai folder baru setiap Cell 2 dijalankan. Path checkpoint diambil dari laporan final yang telah diperiksa; jika folder Drive Anda dipindahkan, sesuaikan `RESEARCH` atau `SOURCES` pada Cell 3. Tidak ada fallback ke bobot acak.

In [ ]:
RESEARCH = Path('/content/drive/MyDrive/Research')
MODELS_TO_RUN = ['vgg16','densenet121','mobilenet_v2','nasnet_mobile']
BATCHES = [1, 8]
WARMUP = 30
TRIALS = 5
ITERATIONS = 50
DOWNLOAD_REPORT = True
assert 1 in BATCHES and len(set(BATCHES))==len(BATCHES) and all(isinstance(b,int) and b>0 for b in BATCHES)
assert WARMUP>=10 and TRIALS>=2 and ITERATIONS>=10
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')+'_'+uuid.uuid4().hex[:6]
OUT = RESEARCH/'outputs/benchmark_4datasets/b08c_efficiency'/RUN_ID
LOCAL = Path('/content')/('b08c_efficiency_'+RUN_ID)
OUT.mkdir(parents=True,exist_ok=False)
LOCAL.mkdir(parents=True,exist_ok=False)
print('Output baru:',OUT)

## Cell 3 — Verifikasi dan salin checkpoint ke disk lokal
Checkpoint Teacher berasal dari B04; checkpoint Student perwakilan berasal dari kontrol B05 fixed-budget. Semua di CBIS-DDSM seed 42. Bobot hanya dibaca. Hash Teacher dicocokkan dengan laporan lama; seluruh file sumber juga di-hash sebelum/sesudah disalin untuk mendeteksi salinan rusak. Hash Student saat ini dicatat sebagai provenance, bukan sebagai verifikasi terhadap hash historis yang tidak tersedia.

In [ ]:
REGISTRY = {'vgg16': ['teacher', 'VGG16', 'vgg16'], 'densenet121': ['teacher', 'DenseNet121', 'densenet'], 'mobilenet_v2': ['student', 'MobileNetV2', 'mobilenet_v2'], 'nasnet_mobile': ['student', 'NASNetMobile', 'nasnet']}
EXPECTED_PARAMETERS = {'vgg16': 14715201, 'densenet121': 7038529, 'mobilenet_v2': 2259265, 'nasnet_mobile': 4270773}
SOURCES = {'vgg16': {'checkpoint': '/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b04_ablation_runs_v1/cbis_ddsm/ab66d3a7f2bd/vgg16/seed_42/finetune/best/ckpt-11', 'expected_hashes': {'ckpt-11.data-00000-of-00001': '678cb3ed61a02092b0d6da844fe7b4bae5104625af24b4eaf8c5bf26b90ddbc4', 'ckpt-11.index': '228cccadd4ac2d4a23740e03b5ff4ebe2d053ff035077ae670d2e7d4eb82a641'}}, 'densenet121': {'checkpoint': '/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b04_ablation_runs_v1/cbis_ddsm/ab66d3a7f2bd/densenet121/seed_42/finetune/best/ckpt-12', 'expected_hashes': {'ckpt-12.data-00000-of-00001': '7364c9f09e06ec886967ba775d28fc3a00320cc9593b241339a20f949611c426', 'ckpt-12.index': '34460c2e0d3f402b4c73cb17712236e4f88f3d3fa164e1f008703bcd3b01c540'}}, 'mobilenet_v2': {'checkpoint': '/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/mobilenet_v2__from__none/seed_42/finetune/best/ckpt-22', 'expected_hashes': {}}, 'nasnet_mobile': {'checkpoint': '/content/drive/MyDrive/Research/outputs/benchmark_4datasets/b05b_inbreast_fixed25_v1/cbis_ddsm/7a4bf6d1e3f6/nasnet_mobile__from__none/seed_42/finetune/best/ckpt-25', 'expected_hashes': {}}}

def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):h.update(block)
    return h.hexdigest()

def save(path,value):
    Path(path).write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8')

assert len(set(MODELS_TO_RUN))==len(MODELS_TO_RUN) and set(MODELS_TO_RUN)<=set(REGISTRY)
CHECKPOINTS={}
for key in MODELS_TO_RUN:
    s=SOURCES[key]
    prefix=Path(s['checkpoint'].replace('/content/drive/MyDrive/Research',str(RESEARCH),1))
    members=[Path(str(prefix)+'.index')]+sorted(prefix.parent.glob(prefix.name+'.data-*'))
    if len(members)<2 or not all(p.is_file() for p in members):
        raise FileNotFoundError(f'{key}: checkpoint tidak lengkap: {prefix}. Cek lokasi Drive; jangan training ulang.')
    dest=LOCAL/'checkpoints'/key;dest.mkdir(parents=True,exist_ok=True)
    hashes={}
    for p in members:
        digest=sha(p)
        expected=s['expected_hashes'].get(p.name)
        if expected and digest!=expected:raise RuntimeError('Hash sumber berubah: '+str(p))
        local=dest/p.name;shutil.copyfile(p,local)
        if sha(local)!=digest:raise RuntimeError('Salinan checkpoint rusak: '+str(local))
        hashes[p.name]=digest
    CHECKPOINTS[key]=dict(original_checkpoint=str(prefix),local_checkpoint=str(dest/prefix.name),checkpoint_sha256=hashes)
    print(key,'checkpoint tersedia')
save(OUT/'checkpoint_provenance.json',CHECKPOINTS)

## Cell 4 — Tulis worker dan protokol
Builder berasal dari `b04_compat.py` dalam laporan B05B. Hanya pembuatan optimizer/compile dihapus, sehingga parameter optimizer tidak terbawa profiling. Forward graph, classifier sigmoid, dan BN non-fused dipertahankan. DenseNet121 diperlakukan sebagai **Teacher** sesuai rancangan KD final, walaupun registry benchmark awal menempatkannya dalam kelompok Student.

In [ ]:
ORIGINAL_BUILDER = "def build_model(key, size=224, seed=42, weights='imagenet', learning_rate=1e-5, stage='finetune'):\n    keras.utils.set_random_seed(seed)\n    _, constructor, _ = REGISTRY[key]\n    base = getattr(keras.applications, constructor)(include_top=False, weights=weights,\n        input_shape=(size,size,3), pooling='avg')\n    base.trainable = stage == 'finetune'\n    for layer in base.layers:\n        if isinstance(layer, keras.layers.BatchNormalization):\n            layer.trainable = False\n            layer.fused = False\n    inp = keras.Input((size,size,3), name='preprocessed_image')\n    x = base(inp, training=False)\n    x = keras.layers.Dropout(.2, seed=seed, name='head_dropout')(x)\n    output = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='probability')(x)\n    model = keras.Model(inp, output, name=key)\n    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate),\n        loss=keras.losses.BinaryCrossentropy())\n    model.optimizer.build(model.trainable_variables)\n    return model"
BUILDER_SOURCE = "def build_model(key, size=224, seed=42, weights='imagenet', learning_rate=1e-05, stage='finetune'):\n    keras.utils.set_random_seed(seed)\n    _, constructor, _ = REGISTRY[key]\n    base = getattr(keras.applications, constructor)(include_top=False, weights=weights, input_shape=(size, size, 3), pooling='avg')\n    base.trainable = stage == 'finetune'\n    for layer in base.layers:\n        if isinstance(layer, keras.layers.BatchNormalization):\n            layer.trainable = False\n            layer.fused = False\n    inp = keras.Input((size, size, 3), name='preprocessed_image')\n    x = base(inp, training=False)\n    x = keras.layers.Dropout(0.2, seed=seed, name='head_dropout')(x)\n    output = keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='probability')(x)\n    model = keras.Model(inp, output, name=key)\n    return model\n"
WORKER_SOURCE = '"""Isolated forward-only profiling; no training and no dataset evaluation."""\nimport os,sys,json,time,hashlib,platform,subprocess,gc\nfrom pathlib import Path\nos.environ.setdefault(\'TF_CPP_MIN_LOG_LEVEL\',\'2\')\nos.environ[\'TF_DETERMINISTIC_OPS\']=\'1\'\nimport numpy as np\nimport tensorflow as tf\nimport tf_keras as keras\n\ndef save(path,obj):\n    Path(path).write_text(json.dumps(obj,indent=2,allow_nan=False),encoding=\'utf-8\')\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for block in iter(lambda:f.read(8*1024*1024),b\'\'):h.update(block)\n    return h.hexdigest()\n\ndef weighted_macs(layer):\n    """Per-image Conv/Dense multiply-accumulates, excluding all other operations."""\n    sh=layer.output_shape\n    if isinstance(sh,list):raise ValueError(\'Multiple outputs not supported: \'+layer.name)\n    dims=sh[1:]\n    if any(x is None for x in dims):raise ValueError(\'Unknown static shape: \'+layer.name)\n    outputs=int(np.prod(dims))\n    L=keras.layers\n    if isinstance(layer,L.SeparableConv2D):\n        kh,kw,inc,mult=map(int,layer.depthwise_kernel.shape)\n        spatial=int(np.prod(dims[:-1]))\n        return spatial*inc*mult*kh*kw + outputs*inc*mult\n    if isinstance(layer,L.DepthwiseConv2D):\n        kh,kw,_,_=map(int,layer.depthwise_kernel.shape)\n        return outputs*kh*kw\n    if isinstance(layer,L.Conv2D):\n        kh,kw,inc_per_group,_=map(int,layer.kernel.shape)\n        return outputs*kh*kw*inc_per_group\n    if isinstance(layer,L.Dense):return outputs*int(layer.kernel.shape[0])\n    raise TypeError(layer.__class__.__name__)\n\ndef count_macs(model):\n    seen=set();rows=[];excluded={}\n    supported=(keras.layers.Conv2D,keras.layers.DepthwiseConv2D,keras.layers.SeparableConv2D,keras.layers.Dense)\n    for layer in model.submodules:\n        if id(layer) in seen or isinstance(layer,keras.Model):continue\n        seen.add(id(layer))\n        if isinstance(layer,supported):\n            if len(layer._inbound_nodes)!=1:\n                raise RuntimeError(\'Shared/called-multiple-times layer needs operation-level count: \'+layer.name)\n            rows.append(dict(layer=layer.name,type=type(layer).__name__,macs=weighted_macs(layer)))\n        else:\n            k=type(layer).__name__;excluded[k]=excluded.get(k,0)+1\n            if layer.weights and not isinstance(layer,keras.layers.BatchNormalization):\n                raise RuntimeError(\'Unaccounted weighted layer: \'+k)\n    if not rows:raise RuntimeError(\'MAC count empty\')\n    return sum(r[\'macs\'] for r in rows),rows,excluded\n\ndef main(config_path):\n    c=json.loads(Path(config_path).read_text());out=Path(c[\'job_dir\']);out.mkdir(parents=True,exist_ok=True)\n    gpus=tf.config.list_physical_devices(\'GPU\')\n    if not gpus:raise RuntimeError(\'GPU required; CPU timing is deliberately rejected.\')\n    tf.config.set_visible_devices(gpus[0],\'GPU\')\n    tf.config.experimental.set_memory_growth(gpus[0],True)\n    tf.config.experimental.enable_tensor_float_32_execution(False)\n    tf.config.experimental.enable_op_determinism()\n    keras.mixed_precision.set_global_policy(\'float32\')\n    keras.utils.set_random_seed(42)\n    # The archived B04 builder is used verbatim except removal of optimizer creation.\n    ns={\'keras\':keras,\'tf\':tf,\'REGISTRY\':c[\'registry\']}\n    exec(compile(Path(c[\'builder_path\']).read_text(),c[\'builder_path\'],\'exec\'),ns)\n    with tf.device(\'/GPU:0\'):\n        model=ns[\'build_model\'](c[\'model\'],size=512,seed=42,weights=None,stage=\'head\')\n    if model.count_params()!=c[\'expected_parameters\']:\n        raise RuntimeError(f"Parameter mismatch: {model.count_params()} != {c[\'expected_parameters\']}")\n    assert model.output_shape==(None,1)\n    assert model.get_layer(\'probability\').activation.__name__==\'sigmoid\'\n    assert not any(\'projector\' in l.name for l in model.submodules)\n    base=next(l for l in model.layers if isinstance(l,keras.Model))\n    ckpt=tf.train.Checkpoint(backbone=base,classifier=model.get_layer(\'probability\'))\n    ckpt.restore(c[\'local_checkpoint\']).assert_existing_objects_matched().expect_partial()\n    model.trainable=False\n    macs,mac_rows,excluded=count_macs(model)\n    save(out/\'macs_by_layer.json\',mac_rows)\n    with open(out/\'model_summary.txt\',\'w\') as f:model.summary(print_fn=lambda line:f.write(line+\'\\n\'),expand_nested=True)\n    weight_bytes=sum(int(np.prod(v.shape))*tf.as_dtype(v.dtype).size for v in model.weights)\n    weight_file=out/\'deployment.weights.h5\'\n    model.save_weights(str(weight_file))\n    h5_bytes=weight_file.stat().st_size\n    weight_file.unlink() # Only the measured size is exported, not private/large checkpoint copies.\n    # Identical synthetic grayscale source distribution; architecture preprocessing is outside timing.\n    rng=np.random.default_rng(42)\n    raw=rng.uniform(0,255,(c[\'batch\'],512,512,1)).astype(np.float32)\n    raw=np.repeat(raw,3,axis=-1)\n    with tf.device(\'/GPU:0\'):\n        x=getattr(keras.applications,c[\'preprocess_module\']).preprocess_input(tf.convert_to_tensor(raw))\n    del raw\n    if \'GPU\' not in x.device:raise RuntimeError(\'Input tensor is not on GPU\')\n    @tf.function(input_signature=[tf.TensorSpec([c[\'batch\'],512,512,3],tf.float32)],jit_compile=False,autograph=False)\n    def forward(v):return model(v,training=False)\n    print(c[\'model\'],\'batch\',c[\'batch\'],\'restored; warm-up\',flush=True)\n    for _ in range(c[\'warmup\']):p=forward(x).numpy()\n    if p.shape!=(c[\'batch\'],1) or not np.isfinite(p).all():raise RuntimeError(\'Invalid forward output\')\n    if not ((p>=0)&(p<=1)).all():raise RuntimeError(\'Invalid probability range\')\n    del p;gc.collect()\n    baseline=tf.config.experimental.get_memory_info(\'GPU:0\')[\'current\']\n    tf.config.experimental.reset_memory_stats(\'GPU:0\')\n    timings=[];trials=[]\n    for trial in range(c[\'trials\']):\n        seconds=[]\n        for iteration in range(c[\'iterations\']):\n            start=time.perf_counter_ns()\n            p=forward(x).numpy() # Waits for GPU output; includes tiny output copy/host invocation.\n            elapsed=(time.perf_counter_ns()-start)/1e9\n            seconds.append(elapsed)\n            timings.append(dict(trial=trial+1,iteration=iteration+1,batch_seconds=elapsed))\n        mean=float(np.mean(seconds))\n        trials.append(dict(trial=trial+1,batch_mean_ms=mean*1000,throughput_images_s=c[\'batch\']/mean))\n        print(c[\'model\'],\'batch\',c[\'batch\'],\'trial\',trial+1,\'mean ms\',round(mean*1000,3),flush=True)\n    memory=tf.config.experimental.get_memory_info(\'GPU:0\')\n    vals=np.array([r[\'batch_seconds\']*1000 for r in timings]);means=np.array([r[\'batch_mean_ms\'] for r in trials])\n    env=dict(python=platform.python_version(),tensorflow=tf.__version__,tf_keras=keras.__version__,\n        gpu=tf.config.experimental.get_device_details(gpus[0]),build=tf.sysconfig.get_build_info(),\n        tf32_enabled=tf.config.experimental.tensor_float_32_execution_enabled(),precision=\'float32\')\n    try:env[\'nvidia_smi\']=subprocess.check_output([\'nvidia-smi\'],text=True)\n    except Exception as e:env[\'nvidia_smi_error\']=str(e)\n    save(out/\'environment.json\',env);save(out/\'timings.json\',timings);save(out/\'trials.json\',trials)\n    result=dict(model=c[\'model\'],role=c[\'role\'],batch=c[\'batch\'],images_per_call=c[\'batch\'],\n        parameters=model.count_params(),parameter_millions=model.count_params()/1e6,\n        conv_dense_macs=macs,conv_dense_gmacs=macs/1e9,conv_dense_gflops_2x_macs=2*macs/1e9,\n        raw_weight_MiB=weight_bytes/2**20,h5_weight_MiB=h5_bytes/2**20,\n        batch_latency_mean_ms=float(vals.mean()),batch_latency_median_ms=float(np.median(vals)),\n        batch_latency_p95_ms=float(np.quantile(vals,.95)),batch_latency_sd_ms=float(vals.std(ddof=1)),\n        trial_mean_latency_sd_ms=float(means.std(ddof=1)),amortized_ms_per_image=float(vals.mean()/c[\'batch\']),\n        throughput_images_s=float(c[\'batch\']*1000/vals.mean()),\n        tf_active_baseline_MiB=baseline/2**20,tf_active_peak_MiB=memory[\'peak\']/2**20,\n        tf_extra_peak_over_baseline_MiB=max(0,memory[\'peak\']-baseline)/2**20,\n        gpu=env[\'gpu\'].get(\'device_name\'),tensorflow=tf.__version__,tf_keras=keras.__version__,\n        tf32_enabled=env[\'tf32_enabled\'],source_dataset=\'cbis_ddsm\',source_seed=42,\n        checkpoint=c[\'original_checkpoint\'],checkpoint_sha256=c[\'checkpoint_sha256\'],\n        excluded_mac_operations=excluded,configuration_sha256=sha(config_path),status=\'complete\')\n    save(out/\'result.json\',result)\n\nif __name__==\'__main__\':\n    try:main(sys.argv[1])\n    except Exception:\n        import traceback\n        traceback.print_exc();sys.exit(1)\n'
BUILDER_PATH=LOCAL/'inference_builder.py';BUILDER_PATH.write_text(BUILDER_SOURCE,encoding='utf-8')
WORKER_PATH=LOCAL/'profile_worker.py';WORKER_PATH.write_text(WORKER_SOURCE,encoding='utf-8')
compile(BUILDER_SOURCE,str(BUILDER_PATH),'exec');compile(WORKER_SOURCE,str(WORKER_PATH),'exec')
(OUT/'inference_builder.py').write_text(BUILDER_SOURCE,encoding='utf-8')
(OUT/'profile_worker.py').write_text(WORKER_SOURCE,encoding='utf-8')
(OUT/'original_build_model.txt').write_text(ORIGINAL_BUILDER,encoding='utf-8')
PROTOCOL=dict(version='b08c_efficiency_v1',run_id=RUN_ID,models=MODELS_TO_RUN,batches=BATCHES,
    image_size=512,precision='float32',tf32=False,seed=42,warmup=WARMUP,trials=TRIALS,iterations=ITERATIONS,
    jit_compile=False,deterministic_ops=True,checkpoint_source='trained CBIS-DDSM seed42 representatives',
    deployment='backbone + GAP + single sigmoid classifier; dropout inactive; no KD projector or optimizer',
    timing='host-observed tf.function forward + small output .numpy synchronization; GPU-resident input',
    excluded_timing='file IO, decoding, resizing, preprocessing, input host-to-device transfer, training, warm-up',
    input='fixed synthetic grayscale RGB tensor; architecture-specific preprocessing outside timing',
    memory='TensorFlow active allocations; isolated process per model/batch; reset peak after warm-up',
    compute='Conv2D/DepthwiseConv2D/SeparableConv2D/Dense MACs per image; 2*MACs is partial FLOPs only',
    builder_sha256=sha(BUILDER_PATH),worker_sha256=sha(WORKER_PATH),
    tensorflow=TF_VERSION,tf_keras=metadata.version('tf-keras'),
    gpu_inventory=subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total,driver_version','--format=csv'],text=True))
save(OUT/'protocol.json',PROTOCOL)
print('Worker siap. Cell 5 menjalankan profiling, bukan training.')

## Cell 5 — Jalankan profiling
Setiap kombinasi model–batch mendapat proses GPU baru. Proses menutup dirinya setelah selesai. Latensi disinkronkan melalui hasil `.numpy()`. Jika gagal, log tetap disimpan; notebook tidak mengisi angka perkiraan atau otomatis mengubah batch size.

In [ ]:
RESULTS=[];ERRORS=[]
save(OUT/'report_status.json',dict(complete=False,status='profiling_in_progress'))
for key in MODELS_TO_RUN:
    for batch in BATCHES:
        tag=f'{key}_batch_{batch}'
        job=LOCAL/'jobs'/tag
        archive=OUT/'jobs'/tag
        # Clear only this run's same job on re-execution; never touch source checkpoints.
        for folder in [job,archive]:
            if folder.exists():shutil.rmtree(folder)
            folder.mkdir(parents=True,exist_ok=True)
        config=dict(model=key,role=REGISTRY[key][0],registry=REGISTRY,preprocess_module=REGISTRY[key][2],
            expected_parameters=EXPECTED_PARAMETERS[key],builder_path=str(BUILDER_PATH),job_dir=str(job),
            batch=batch,warmup=WARMUP,trials=TRIALS,iterations=ITERATIONS,**CHECKPOINTS[key])
        cfg=job/'config.json';save(cfg,config)
        result_path=job/'result.json'
        if result_path.exists():result_path.unlink()
        print('START',tag,flush=True)
        with (job/'worker.log').open('w') as log:
            proc=subprocess.Popen([sys.executable,'-u',str(WORKER_PATH),str(cfg)],
                stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            try:
                for line in proc.stdout:
                    log.write(line);log.flush()
                    if any(t in line for t in ['restored;','trial','Error','Traceback','failed']):print(line.rstrip(),flush=True)
                rc=proc.wait()
            except BaseException:
                proc.terminate();proc.wait();raise
        for p in job.iterdir():
            if p.is_file() and p.suffix in ['.json','.txt','.log']:shutil.copyfile(p,archive/p.name)
        if rc==0 and result_path.exists():
            result=json.loads(result_path.read_text())
            if result['configuration_sha256']!=sha(cfg):raise RuntimeError('Config/result mismatch')
            RESULTS.append(result)
        else:
            ERRORS.append(dict(model=key,batch=batch,returncode=rc,log=str(archive/'worker.log')))
            print('FAILED:',tag,'— lihat worker.log; hasil lain dilanjutkan.')
        save(OUT/'partial_results.json',RESULTS);save(OUT/'errors.json',ERRORS)
print('Selesai:',len(RESULTS),'dari',len(MODELS_TO_RUN)*len(BATCHES),'kombinasi.')

## Cell 6 — Tabel efisiensi dan perbandingan Teacher–Student
`GMACs` hanya menghitung perkalian-akumulasi Conv/Dense. `2 × GMACs` **bukan total FLOPs graph** karena BN, pooling, aktivasi, penjumlahan residual, dan bias tidak dihitung. Laporkan konvensi ini di paper.

Latensi batch 1 = waktu satu panggilan citra tunggal. Waktu per citra pada batch 8 bersifat amortisasi dan jangan disebut latensi citra tunggal. Memori adalah alokasi aktif TensorFlow, bukan seluruh memori proses yang terlihat di `nvidia-smi`. SD waktu menunjukkan variasi pengukuran, bukan SD training multiseed.

In [ ]:
if not RESULTS:
    save(OUT/'report_status.json',dict(complete=False,status='all_jobs_failed',errors=ERRORS))
    failure_zip=OUT/'B08C_failure_report.zip'
    with zipfile.ZipFile(failure_zip,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUT.rglob('*'):
            if p.is_file() and p.suffix!='.zip':z.write(p,p.relative_to(OUT))
    if DOWNLOAD_REPORT:files.download(str(failure_zip))
    raise RuntimeError('Belum ada hasil valid. Kirim B08C_failure_report.zip untuk diagnosis.')
raw=pd.DataFrame(RESULTS)
if raw[['gpu','tensorflow','tf_keras','tf32_enabled']].drop_duplicates().shape[0]!=1:
    raise RuntimeError('Hasil berasal dari lingkungan berbeda; ulangi profiling dalam runtime yang sama.')
for key,g in raw.groupby('model'):
    for col in ['parameters','conv_dense_macs','raw_weight_MiB']:
        if g[col].nunique()!=1:raise RuntimeError(f'{key}: {col} tidak konsisten antarbatch')
rows=[]
throughput_batch=max(BATCHES)
for key in MODELS_TO_RUN:
    one=raw[(raw.model==key)&(raw.batch==1)]
    many=raw[(raw.model==key)&(raw.batch==throughput_batch)]
    if one.empty or many.empty:continue
    a=one.iloc[0];b=many.iloc[0]
    rows.append(dict(model=key,role=a.role,parameters=a.parameters,parameters_M=a.parameter_millions,
        conv_dense_GMACs=a.conv_dense_gmacs,weights_H5_MiB=a.h5_weight_MiB,
        latency_B1_mean_ms=a.batch_latency_mean_ms,latency_B1_median_ms=a.batch_latency_median_ms,
        latency_B1_p95_ms=a.batch_latency_p95_ms,latency_B1_trial_mean_SD_ms=a.trial_mean_latency_sd_ms,
        throughput_batch=throughput_batch,throughput_images_s=b.throughput_images_s,
        peak_active_B1_MiB=a.tf_active_peak_MiB,peak_active_throughput_batch_MiB=b.tf_active_peak_MiB,gpu=a.gpu))
summary=pd.DataFrame(rows)
if summary.empty:raise RuntimeError('Profiling parsial belum memiliki pasangan batch lengkap.')
ratio_rows=[]
for teacher in ['vgg16','densenet121']:
    for student in ['mobilenet_v2','nasnet_mobile']:
        t=summary[summary.model==teacher];s=summary[summary.model==student]
        if t.empty or s.empty:continue
        t=t.iloc[0];s=s.iloc[0]
        ratio_rows.append(dict(teacher=teacher,student=student,
            parameter_reduction_percent=100*(1-s.parameters/t.parameters),
            conv_dense_MAC_reduction_percent=100*(1-s.conv_dense_GMACs/t.conv_dense_GMACs),
            latency_B1_speedup=t.latency_B1_mean_ms/s.latency_B1_mean_ms))
ratios=pd.DataFrame(ratio_rows)
COMPLETE=(not ERRORS and len(RESULTS)==len(MODELS_TO_RUN)*len(BATCHES) and set(MODELS_TO_RUN)==set(REGISTRY))
save(OUT/'report_status.json',dict(complete=COMPLETE,expected=len(MODELS_TO_RUN)*len(BATCHES),
    successful=len(RESULTS),errors=ERRORS,training_performed=False,test_evaluated=False))
raw.to_csv(OUT/'all_batch_results.csv',index=False)
summary.to_csv(OUT/'efficiency_summary.csv',index=False)
ratios.to_csv(OUT/'teacher_student_ratios.csv',index=False)
display(summary.round(4));display(ratios.round(4))
print('COMPLETE' if COMPLETE else 'PARTIAL — jangan gunakan sebagai tabel lengkap paper; lihat error/model yang belum dijalankan.')

## Cell 7 — Simpan Excel dan ZIP untuk ditinjau
Excel berisi tabel ringkas, seluruh batch, rasio Teacher–Student, dan catatan metode. ZIP memuat protokol, hash, audit MAC per layer, log, raw timing, dan hasil. Checkpoint tidak ikut ZIP. Pengukuran tidak menyentuh CSV atau checkpoint penelitian lama.

In [ ]:
notes=[
    ('Status','COMPLETE' if COMPLETE else 'PARTIAL: inspect errors.json'),
    ('Scope','Inference efficiency only; no training or test evaluation'),
    ('Architecture','B04/B05 builder; optimizer creation removed; backbone + GAP + sigmoid head'),
    ('KD','Student architecture unchanged across control/Hinton/feature KD; teacher/projector excluded'),
    ('Weights','Trained CBIS-DDSM seed42 representative checkpoints; not all KD variants reprofiled'),
    ('MACs','Conv/Dense only, per image; BN, pooling, activations, residual adds and biases excluded'),
    ('FLOPs','2 x MACs is Conv/Dense FLOPs convention, NOT total graph FLOPs'),
    ('Latency','Host-observed synchronized forward; GPU-resident synthetic input; small output copy included'),
    ('Excluded time','Image reading, resizing, normalization, input transfer, compilation and warm-up'),
    ('Precision','float32; TF32 disabled; XLA disabled; deterministic ops enabled'),
    ('Memory','TensorFlow active allocation peak after warm-up reset; isolated subprocess per model/batch'),
    ('SD','Variation of repeated timing; not model-training seed variation'),
    ('Comparisons','Same GPU/runtime/batch/precision required; reductions are architecture comparisons'),
    ('Training cost','Not measured here; teacher cache/calibration and training costs require separate accounting'),
    ('Sources','https://www.tensorflow.org/api_docs/python/tf/config/experimental/get_memory_info'),
    ('Sources','https://www.tensorflow.org/api_docs/python/tf/config/experimental/reset_memory_stats'),
]
xlsx=OUT/'KD_efficiency_results.xlsx'
with pd.ExcelWriter(xlsx,engine='openpyxl') as writer:
    summary.to_excel(writer,sheet_name='Efficiency',index=False)
    raw.drop(columns=['checkpoint_sha256','excluded_mac_operations']).to_excel(writer,sheet_name='All batches',index=False)
    ratios.to_excel(writer,sheet_name='Teacher Student ratios',index=False)
    pd.DataFrame(notes,columns=['Item','Description']).to_excel(writer,sheet_name='Read me',index=False)
    from openpyxl.styles import Font,PatternFill,Alignment
    for ws in writer.book.worksheets:
        ws.freeze_panes='A2';ws.auto_filter.ref=ws.dimensions
        for cell in ws[1]:
            cell.font=Font(bold=True,color='FFFFFF');cell.fill=PatternFill('solid',fgColor='24476A')
            cell.alignment=Alignment(wrap_text=True,vertical='center')
        ws.row_dimensions[1].height=36
        for col in ws.columns:
            letter=col[0].column_letter
            ws.column_dimensions[letter].width=min(55,max(16,max(len(str(c.value or '')) for c in col)+2))
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment=Alignment(vertical='top',wrap_text=True)
                if isinstance(cell.value,float):cell.number_format='0.0000'
zip_path=OUT/f'B08C_KD_efficiency_{RUN_ID}_report.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if p.is_file() and p!=zip_path:z.write(p,p.relative_to(OUT))
print('Excel:',xlsx);print('ZIP:',zip_path)
print('Kirim ZIP ini untuk pemeriksaan dan memasukkan angka efisiensi ke paper.')
if DOWNLOAD_REPORT:files.download(str(zip_path))

### Batas interpretasi
- Hasil ini mengukur biaya arsitektur inferensi; ukuran arsitektur yang sama berlaku lintas dataset jika input dan classifier sama.
- Rasio kompresi parameter dibandingkan terhadap Teacher tertentu, bukan akibat pengurangan parameter oleh loss KD.
- Jangan menggunakan benchmark GPU untuk mengklaim kecepatan CPU/ponsel tanpa pengukuran perangkat tersebut.
- Jangan menyebut KD efektif hanya karena Student lebih kecil. Gabungkan tabel ini dengan F1, precision, recall, macro-F1, dan AUC pada dataset yang bersangkutan.
- Tidak ada angka efisiensi yang ditanam dalam notebook. Semua angka waktu dan memori berasal dari runtime yang menjalankannya.